In [1]:
import itertools as itt
import pandas as pd
import seaborn as sns

from training_configuration import L1_INFLATE_REGS, LATENT_DIMS, CONTEXTS

from mEncoder.pretraining import generate_cross_sample_pretraining_problem
from mEncoder.analysis import (
    process_simulation,
    load_mae,
    plot_loss_vs_regularization,
    load_optimize_result_pretraining_cross_samples,
    evaluate_simulations,
)
from mEncoder import (
    pretrain_dir,
    data_dir,
    fig_dir,
    apply_objective_settings,
)

MODEL = "EGFR"
DATA = "dream_cytof"
SAMPLES = "0_5"

%matplotlib inline

In [2]:
xx = []
for l1reg, latent_dim, context in itt.product(L1_INFLATE_REGS, LATENT_DIMS, CONTEXTS):
    mae = load_mae("train", DATA, MODEL, context, SAMPLES, latent_dim, l1reg)

    problem_cross_sample = generate_cross_sample_pretraining_problem(mae)
    result = load_optimize_result_pretraining_cross_samples(
        MODEL, DATA, context, SAMPLES, latent_dim, l1reg
    )

    x = problem_cross_sample.objective.infun(result.list[0]["x"])
    for name, xi in zip(mae.pypesto_subproblem.x_names, x):
        if not name.startswith("INPUT_"):
            continue
        line = name[6:].split("__")[-1]
        par = "__".join(name[6:].split("__")[:-1])
        xx.append(
            dict(
                l1reg=l1reg,
                n_latent_dim=latent_dim,
                par=par,
                cell_line=line,
                x=xi,
                context=context,
            )
        )

In [6]:
df = pd.DataFrame(xx)
g = sns.FacetGrid(df, col="n_latent_dim", row="par")
g.map_dataframe(sns.lineplot, x="l1reg", y="x", hue="context")
g.set(xscale="log")